# Tarea 3 
INF-452 · Geometría Computacional

**Integrantes:**
* Andrés Jablonca 202173578-K
* Benjamín Olguín 202273517-1

Ejecutar las celdas en orden con la carpeta `data/` en la misma carpeta del ipynb. Se requiere Python 3 y NumPy.

## (a) Orientación

El determinante $D=(x_2-x_1)(y_3-y_1)-(y_2-y_1)(x_3-x_1)$ es positivo a la izquierda de la arista dirigida, negativo a la derecha y cero cuando los puntos son colineales. La función devuelve el signo inverso, como pide el enunciado. El resultado 0 incluye la prolongación de la arista, ya que la clasificación usa la recta que la contiene. Para coordenadas de punto flotante, se trata como cero un determinante dentro de una tolerancia relativa a los productos de la resta.

In [1]:
import numpy as np

TOLERANCIA = 8 * np.finfo(np.float64).eps


def _orientacion(dx, dy, x, y):
    producto_1 = dx * y
    producto_2 = dy * x
    determinante = producto_1 - producto_2
    margen = TOLERANCIA * (np.abs(producto_1) + np.abs(producto_2))
    return np.where(determinante > margen, -1,
                    np.where(determinante < -margen, 1, 0))


def check_segment(p1, p2, p3):
    p1, p2, p3 = np.asarray([p1, p2, p3], dtype=np.float64)
    dx, dy = p2 - p1
    x, y = p3 - p1
    return int(_orientacion(dx, dy, x, y))

## (b) Punto en triángulo

Un triángulo no degenerado es la intersección de los tres semiplanos interiores de sus aristas. Por eso el punto está dentro si las tres orientaciones son no positivas o las tres son no negativas. Las igualdades incluyen el borde y la doble condición permite ambos órdenes de vértices. Se supone que los triángulos de la subdivisión tienen área positiva.

In [2]:
def check_triangle(T, q):
    a, b, c = T
    lados = (check_segment(a, b, q), check_segment(b, c, q),
             check_segment(c, a, q))
    dentro = max(lados) <= 0 or min(lados) >= 0
    return 1 if dentro else -1

## (c) Búsqueda lineal

Se preparan las aristas de los $m$ triángulos. Por cada consulta se evalúan las mismas tres orientaciones para todos los triángulos con operaciones vectorizadas de NumPy. Como la entrada está en orden antihorario, basta con exigir que la consulta quede a la izquierda o sobre la recta de cada arista. Si el punto toca varias caras, se devuelve el menor índice de fila, con numeración desde cero.

La preparación cuesta $O(m)$. Cada consulta revisa $m$ triángulos, con tres aristas por triángulo, así que cuesta $\Theta(m)$ y usa $O(m)$ de memoria auxiliar. Para $r$ consultas el tiempo total es $O(m+rm)$ y el vector de salida ocupa $O(r)$. En una triangulación planar de $n$ vértices, $m=O(n)$ por la fórmula de Euler, así que el tiempo por consulta es $O(n)$. Para una triangulación del casco convexo con $h$ vértices en su frontera, $m=2n-2-h$.

In [3]:
def linear_point_location(P, T, Q):
    vertices = np.asarray(P, dtype=np.float64)[T]
    aristas = np.roll(vertices, -1, axis=1) - vertices
    x, y = vertices[:, :, 0], vertices[:, :, 1]
    dx, dy = aristas[:, :, 0], aristas[:, :, 1]

    indices = np.full(len(Q), -1, dtype=np.intp)
    if len(T) == 0:
        return indices

    for j, (qx, qy) in enumerate(Q):
        lados = _orientacion(dx, dy, qx - x, qy - y)
        dentro = np.all(lados <= 0, axis=1)
        candidato = dentro.argmax()
        if dentro[candidato]:
            indices[j] = candidato

    return indices

In [4]:
P = np.array([[0, 0], [2, 0], [2, 2], [0, 2]])
T = np.array([[0, 1, 2], [0, 2, 3]])
Q = np.array([[1.5, 0.5], [0.5, 1.5], [1, 1], [0, 0], [3, 3]])

assert check_segment(P[0], P[1], [1, 1]) == -1
assert check_segment(P[0], P[1], [1, -1]) == 1
assert check_segment(P[0], P[1], [1, 0]) == 0
assert check_triangle(P[T[0]], [1, 1]) == 1
assert check_triangle(P[T[0][::-1]], [1.5, 0.5]) == 1
assert check_triangle(P[T[0]], [3, 0]) == -1
np.testing.assert_array_equal(linear_point_location(P, T, Q), [0, 1, 0, 0, -1])
np.testing.assert_array_equal(linear_point_location(P, T[:0], Q), [-1] * len(Q))
print("Pruebas de orientación, interior, borde y exterior superadas.")

Pruebas de orientación, interior, borde y exterior superadas.


## (d) Experimento

Se miden las 12 combinaciones del enunciado. Aquí $r$ denota el número de consultas y $m$ el número de triángulos. Cada tiempo es la mediana de tres ejecuciones completas después de una consulta de calentamiento. La carga de archivos queda fuera del cronómetro y la preparación de las aristas queda dentro. Para comparar tamaños, se divide el tiempo por $r$ y luego por $m$.

In [5]:
from statistics import median
from time import perf_counter
from pathlib import Path

datos = {ruta.name: np.load(ruta) for ruta in Path("data").glob("*.npy")}

resultados = []
print(f"{'n':>7} {'m':>7} {'r':>6} {'segundos':>10} "
      f"{'µs/consulta':>13} {'ns/(consulta·triángulo)':>24} {'fuera':>6}")
for n in (100, 1000, 10000, 50000):
    P = datos[f"points_{n}.npy"]
    T = datos[f"triangles_{n}.npy"]
    m = len(T)
    for r in (100, 1000, 5000):
        Q = datos[f"query_{r}.npy"]
        linear_point_location(P, T, Q[:1])
        tiempos = []
        for _ in range(3):
            inicio = perf_counter()
            indices = linear_point_location(P, T, Q)
            tiempos.append(perf_counter() - inicio)
        segundos = median(tiempos)

        # Referencia escalar para una muestra fija de cada combinación.
        for j in np.linspace(0, r - 1, 3, dtype=int):
            esperado = next((i for i, fila in enumerate(T)
                             if check_triangle(P[fila], Q[j]) == 1), -1)
            assert indices[j] == esperado

        resultados.append((n, m, r, segundos))
        print(f"{n:7d} {m:7d} {r:6d} {segundos:10.4f} "
              f"{1e6 * segundos / r:13.2f} {1e9 * segundos / (r * m):24.2f} "
              f"{np.count_nonzero(indices == -1):6d}")

      n       m      r   segundos   µs/consulta  ns/(consulta·triángulo)  fuera
    100     186    100     0.0026         25.79                   138.65     36
    100     186   1000     0.0291         29.14                   156.67    318
    100     186   5000     0.1427         28.55                   153.49   1708
   1000    1973    100     0.0091         91.44                    46.34     24
   1000    1973   1000     0.1008        100.76                    51.07    238
   1000    1973   5000     0.4671         93.42                    47.35   1279
  10000   19939    100     0.1670       1669.73                    83.74     21
  10000   19939   1000     1.0057       1005.70                    50.44    212
  10000   19939   5000     4.3393        867.87                    43.53   1113
  50000   99893    100     1.2810      12809.84                   128.24     21
  50000   99893   1000     9.4333       9433.25                    94.43    206
  50000   99893   5000    48.1601       

In [6]:
print("Pendiente del ajuste log(tiempo por consulta) vs. log(m)")
for r in (100, 1000, 5000):
    muestra = [(m, tiempo / r) for _, m, consultas, tiempo in resultados
               if consultas == r]
    triangulos, tiempos_por_consulta = np.asarray(muestra).T
    pendiente, _ = np.polyfit(np.log(triangulos), np.log(tiempos_por_consulta), 1)
    print(f"r = {r:4d}   pendiente = {pendiente:.3f}")

Pendiente del ajuste log(tiempo por consulta) vs. log(m)
r =  100   pendiente = 1.007
r = 1000   pendiente = 0.914
r = 5000   pendiente = 0.915


## Conclusión

Las pendientes medidas para 100, 1.000 y 5.000 consultas fueron 1,007, 0,914 y 0,915, respectivamente. Son cercanas a 1 y compatibles con el crecimiento lineal del tiempo por consulta. El costo por triángulo varió entre 43,53 y 156,67 ns, por lo que la proporcionalidad no es exacta en los tamaños medidos. La cota $O(n)$ se obtiene del análisis de la búsqueda y de la relación $m=O(n)$ en una triangulación planar. En los 12 experimentos, las tres consultas muestreadas coincidieron con la búsqueda escalar de referencia.
